In [0]:
%python
%pip install pyiceberg[pyarrow]
%restart_python

In [0]:
%run ./Classroom-Setup-Common

In [0]:
-- Preflight: ensure 0 - Required Setup has been run for this user
BEGIN
  DECLARE schema_exists BOOLEAN DEFAULT FALSE;

  SET schema_exists = (
    SELECT COUNT(*) > 0
    FROM system.information_schema.schemata
    WHERE catalog_name = current_catalog()
      AND schema_name = 'data_interoperability_tpcds'
  );

  IF NOT schema_exists THEN
    SELECT raise_error(
      'Schema "' || current_catalog() || '.data_interoperability_tpcds" not found. ' ||
      'Run the "0 - Required Setup" notebook before running this notebook.'
    );
  END IF;
END;

In [0]:
USE SCHEMA data_interoperability_tpcds;

In [0]:
-- Create the lab's plain Delta baseline from the TPC-DS item dimension (~300K rows).
-- The lab will enable Iceberg V3 reads on this table and observe deletion-vector behaviour.
-- DROP first so re-running this setup is safe even if a prior run upgraded the table to IcebergCompat
-- (CREATE OR REPLACE alone refuses to downgrade an IcebergCompat-enabled table back to plain Delta).
DROP TABLE IF EXISTS item_demo;

CREATE TABLE item_demo
AS SELECT * FROM samples.tpcds_sf1000.item;

In [0]:
SELECT
  current_catalog() AS catalog,
  current_schema()  AS schema,
  (SELECT COUNT(*) FROM item_demo) AS item_demo_rows;